In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\TANVEER\Documents\Altamash\Funnel A_B testing\Data\SQL_Query_Results\session_metrics.csv"
)

np.random.seed(42)

df["experiment_group"] = np.random.choice(
    ["control", "treatment"],
    size=len(df),
    p=[0.5, 0.5]
)

df["experiment_conversion"] = df["purchased"]

In [9]:
df.head(10)


,user_session,user_id,session_start,session_duration_seconds,viewed,added_to_cart,purchased,revenue,product_count,experiment_group,experiment_conversion
0,18af673b-7fb9-4202-a66d-5c855bc0fd2d,619841242,2020-03-01 05:29:28,26,2,0,0,0.0,2,control,0
1,cc8a2df5-2541-453d-a465-b0b9f888e71c,325496098,2020-03-01 05:29:04,0,1,0,0,0.0,1,treatment,0
2,07c2bdcb-a37c-4444-85b3-c3e3089975e8,577096637,2020-03-01 05:28:26,13,0,2,0,0.0,2,treatment,0
3,fd47f5cd-6ed1-43f1-a9c6-df6f0f72538a,607916558,2020-03-01 05:28:24,0,1,0,0,0.0,1,treatment,0
4,268c0dae-3755-469d-bc7c-8e56f37888d9,607916558,2020-03-01 05:28:20,0,1,0,0,0.0,1,control,0
5,062d2f70-2039-4e9d-8530-ec12874e8cf0,622090052,2020-03-01 05:27:24,0,1,0,0,0.0,1,control,0
6,ab7d349f-db5d-4790-8ab1-31e5c894459d,622090043,2020-03-01 05:27:23,142,2,1,0,0.0,3,control,0
7,10316cd5-adef-486d-9915-c2ca7870c8e8,548866558,2020-03-01 05:27:21,12,0,3,0,0.0,1,treatment,0
8,a859392f-9169-4f8a-b761-0e106f666259,621899361,2020-03-01 05:26:29,0,1,0,0,0.0,1,treatment,0
9,b395bede-b451-41ca-bd41-7fb90b92813e,610682174,2020-03-01 05:26:22,0,1,0,0,0.0,1,treatment,0


In [7]:
df["experiment_conversion"].value_counts()

experiment_conversion
0      4380654
3        17082
2        16853
1        15855
4        15088
        ...   
202          1
192          1
156          1
193          1
282          1
Name: count, Length: 179, dtype: int64

In [10]:
baseline_rate = df.loc[
    df["experiment_group"] == "control",
    "purchased"
].mean()

target_lift = 0.03

treatment_probability = min(
    baseline_rate * (1 + target_lift),
    0.99
)

In [14]:
print(baseline_rate) 
print(treatment_probability)

0.2843032671625435
0.29283236517741984


In [17]:
treatment_users = (
    df["experiment_group"] == "treatment"
)

df.loc[treatment_users, "experiment_conversion"] = (
    np.random.binomial(
        1,
        treatment_probability,
        treatment_users.sum()
    )
)

# Verify actual observed conversion rates in both groups

actual_control_cr = df.loc[df["experiment_group"] == "control", "experiment_conversion"].mean()
actual_treatment_cr = df.loc[df["experiment_group"] == "treatment", "experiment_conversion"].mean()

print(f"Control Conversion Rate:   {actual_control_cr:.4f}")
print(f"Treatment Conversion Rate: {actual_treatment_cr:.4f}")
print(f"Observed Relative Lift:    {((actual_treatment_cr - actual_control_cr) / actual_control_cr):.2%}")

Control Conversion Rate:   0.2843
Treatment Conversion Rate: 0.2932
Observed Relative Lift:    3.12%


In [12]:
df.head(10)

,user_session,user_id,session_start,session_duration_seconds,viewed,added_to_cart,purchased,revenue,product_count,experiment_group,experiment_conversion
0,18af673b-7fb9-4202-a66d-5c855bc0fd2d,619841242,2020-03-01 05:29:28,26,2,0,0,0.0,2,control,0
1,cc8a2df5-2541-453d-a465-b0b9f888e71c,325496098,2020-03-01 05:29:04,0,1,0,0,0.0,1,treatment,0
2,07c2bdcb-a37c-4444-85b3-c3e3089975e8,577096637,2020-03-01 05:28:26,13,0,2,0,0.0,2,treatment,0
3,fd47f5cd-6ed1-43f1-a9c6-df6f0f72538a,607916558,2020-03-01 05:28:24,0,1,0,0,0.0,1,treatment,0
4,268c0dae-3755-469d-bc7c-8e56f37888d9,607916558,2020-03-01 05:28:20,0,1,0,0,0.0,1,control,0
5,062d2f70-2039-4e9d-8530-ec12874e8cf0,622090052,2020-03-01 05:27:24,0,1,0,0,0.0,1,control,0
6,ab7d349f-db5d-4790-8ab1-31e5c894459d,622090043,2020-03-01 05:27:23,142,2,1,0,0.0,3,control,0
7,10316cd5-adef-486d-9915-c2ca7870c8e8,548866558,2020-03-01 05:27:21,12,0,3,0,0.0,1,treatment,1
8,a859392f-9169-4f8a-b761-0e106f666259,621899361,2020-03-01 05:26:29,0,1,0,0,0.0,1,treatment,1
9,b395bede-b451-41ca-bd41-7fb90b92813e,610682174,2020-03-01 05:26:22,0,1,0,0,0.0,1,treatment,0


In [16]:
print(treatment_users)

0          False
1           True
2           True
3           True
4          False
           ...  
4536266     True
4536267     True
4536268     True
4536269     True
4536270     True
Name: experiment_group, Length: 4536271, dtype: bool


H0:
Traffic is split according to the expected ratio.

H1:
Traffic distribution differs from expected ratio.

### Sample ratio mismatch:

In [21]:
# Sample Ratio Mismatch (SRM) Test

from scipy.stats import chisquare

control_count = (df["experiment_group"] == "control").sum()
treatment_count = (df["experiment_group"] == "treatment").sum()

observed = [control_count, treatment_count]
total = sum(observed)
expected = [total * 0.5, total * 0.5]

stat, p_value = chisquare(observed, expected)
print(f"\nSRM Test:")
print(f"Chi-square Statistic: {stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.01:
    print("WARNING: SRM detected! Sample split differs significantly from 50/50.")
else:
    print("SRM check passed: Assignment ratio aligns with the expected 50/50 split.")


SRM Test:
Chi-square Statistic: 0.1696
P-value: 0.6805
SRM check passed: Assignment ratio aligns with the expected 50/50 split.


### Conversion Test:

In [22]:
from statsmodels.stats.proportion import proportions_ztest

control = df[
    df["experiment_group"] == "control"
]

treatment = df[
    df["experiment_group"] == "treatment"
]

conversions = [
    control["experiment_conversion"].sum(),
    treatment["experiment_conversion"].sum()
]

sample_sizes = [
    len(control),
    len(treatment)
]

z_stat, p_value = proportions_ztest(
    conversions,
    sample_sizes
)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: -20.862210444475604
P-value: 1.1808579785799296e-96
